# Artifact 24: integer-scaled mapped-area periods
## Red and Abel–Wick families, with direct area and mesh verification

This notebook is an **extension**, not a replacement, of two existing project artifacts:

- [`paper/jacobian_counterexample_and_elliptic_integrals.pdf`](../paper/jacobian_counterexample_and_elliptic_integrals.pdf), especially the distinction between flat pre-image area and stretched mapped area;
- [`notebooks/artifact24_executable_paper.ipynb`](artifact24_executable_paper.ipynb), which establishes the triangle map, the red oscillation disk, the mapped-area density, and the original interactive Hamilton–Abel picture.

The new computation adds the three bounded Abel–Wick projections and asks an arithmetic question about the mapped-area **period** series.  All long coefficients live in an ordinary CSV under `data/`.  Before that cache is used, a short exact prefix is recomputed visibly from the polynomial map and Gram determinant.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys, math, warnings, runpy

import numpy as np
import pandas as pd
import sympy as sp
import mpmath as mp
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from IPython.display import display

# Locate the existing Artifact 24 project from either the repository root or
# the notebooks directory used by Binder.
HERE = Path.cwd().resolve()
CANDIDATES = [HERE, HERE.parent, *HERE.parents]
ROOT = next(
    (
        candidate
        for candidate in CANDIDATES
        if (candidate / "artifact24" / "geometry.py").is_file()
        and (candidate / "original_scripts" / "compute_mesh_area_series.py").is_file()
        and (candidate / "paper" / "jacobian_counterexample_and_elliptic_integrals.pdf").is_file()
    ),
    None,
)
if ROOT is None:
    raise RuntimeError("Run this notebook from 24-jupyter-testing/ or its notebooks/ directory.")

sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "original_scripts"))
DATA = ROOT / "data"

# Numerical poles must be masked before division.  Any remaining divide,
# invalid, or overflow warning is treated as an error.
warnings.simplefilter("error", RuntimeWarning)
_OLD_NUMPY_ERRORS = np.seterr(
    divide="raise", invalid="raise", over="raise", under="ignore"
)
mp.mp.dps = 80

print("Artifact 24 root:", ROOT)
print("Paper:", ROOT / "paper" / "jacobian_counterexample_and_elliptic_integrals.pdf")

## 1. Bounded action–angle and Gram-matrix formulation

On each bounded source-plane disk use

$$
X=r\cos\phi,\qquad Y=r\sin\phi,\qquad r=\sqrt{2\lambda},
$$

so that

$$
dX\wedge dY=d\lambda\wedge d\phi.
$$

For the plane embedding $E_i$ and polynomial map $F$, define

$$
R_i(X,Y)=F(E_i(X,Y)),
\qquad
G_i(X,Y)=D R_i(X,Y)^T D R_i(X,Y).
$$

The mapped Euclidean area density is

$$
J_i(X,Y)=\sqrt{\det G_i(X,Y)}
       =\left\|R_{i,X}\times R_{i,Y}\right\|.
$$

Normalize mapped area and its shell derivative by

$$
\Psi_i(s)=\frac{\mathcal A_i(s)}{\pi J_i(0)},
\qquad
\Phi_i(s)=\Psi_i'(s)=\sum_{k\ge0} b_{i,k}s^k.
$$

The arithmetic question is whether one constant $C_i$ makes

$$
A_{i,k}=C_i^k b_{i,k}\in\mathbb Z.
$$

The red calculation continues the exact homogeneous expansion already used in `compute_mesh_area_series.py`.  The Abel–Wick calculation uses the same strategy after a bounded coordinate rationalization.

## 2. Exact homogeneous-density recurrence

After rationalization, write the normalized Gram determinant and its square root as homogeneous sums

$$
Q_i=1+q_{i,1}+q_{i,2}+\cdots,
\qquad
\frac{J_i}{J_i(0)}=1+j_{i,1}+j_{i,2}+\cdots.
$$

For the Abel–Wick action–angle normalization, the finite recurrence implemented in `original_scripts/abel_wick_period_series.py` is

$$
\boxed{
 j_{i,n}=-\frac{1}{2n}
 \sum_{k=1}^{\min(D,n)}(2n-3k)q_{i,k}j_{i,n-k}
}.
$$

The surviving angular monomials are integrated exactly by beta moments.  No finite-field or modular reconstruction is used.

In [ ]:
from compute_mesh_area_series import compute as compute_red_area_series
from abel_wick_period_series import (
    FAMILY_NAMES as ABEL_WICK_NAMES,
    SCALES as ABEL_WICK_SCALES,
    period_coefficients as compute_abel_wick_period,
)

SCALES = {
    "red": sp.Integer(235_651_734),
    **ABEL_WICK_SCALES,
}

# Binder-safe live computation.  Increase deliberately when studying the
# derivation; the full 90/100-term results are checked from the transparent CSV.
LIVE_TERMS = 8

red_data = compute_red_area_series(LIVE_TERMS)
red_area_coefficients = red_data["coeffs"]
live = {
    "red": [
        sp.factor((k + 1) * red_area_coefficients[k + 1])
        for k in range(LIVE_TERMS)
    ]
}
center_gram = {"red": sp.factor(red_data["q0"])}

for family in ABEL_WICK_NAMES:
    coefficients, q0 = compute_abel_wick_period(family, LIVE_TERMS)
    live[family] = coefficients
    center_gram[family] = sp.factor(q0)

rows = []
for family, coefficients in live.items():
    scale = SCALES[family]
    for k, coefficient in enumerate(coefficients):
        scaled = sp.cancel(coefficient * scale**k)
        rows.append({
            "family": family,
            "k": k,
            "period coefficient b_k": str(coefficient),
            "scaled integer A_k": str(scaled),
            "integer": sp.denom(scaled) == 1,
        })

live_table = pd.DataFrame(rows)
display(pd.DataFrame({
    "family": list(center_gram),
    "det G_i(0)": [str(center_gram[name]) for name in center_gram],
    "scale C_i": [str(SCALES[name]) for name in center_gram],
}))
display(live_table)

assert live_table["integer"].all()
assert live["yellow"] == live["blue"]
print("PASS: live exact rational computation, including independent yellow/blue runs")

## 3. Long transparent coefficient cache

The cache `data/mapped_area_period_scaled_integers.csv` contains:

- 90 exact red period coefficients;
- 100 exact coefficients for each of green, yellow, and blue.

It is plain rational/decimal text generated by `original_scripts/build_mapped_area_period_cache.py`.  The live prefix above must agree exactly before any cached coefficient is used.

In [ ]:
cache_path = DATA / "mapped_area_period_scaled_integers.csv"
cache = pd.read_csv(cache_path, dtype=str)
cache["k"] = cache["k"].astype(int)

for family, coefficients in live.items():
    stored = cache[cache.family == family].sort_values("k")
    for k, coefficient in enumerate(coefficients):
        assert sp.Rational(stored.iloc[k].period_coefficient) == coefficient
        assert sp.Integer(stored.iloc[k].scaled_integer) == sp.cancel(
            coefficient * SCALES[family]**k
        )

assert cache.is_integer.eq("true").all()
assert (
    cache[cache.family == "yellow"].sort_values("k").scaled_integer.tolist()
    == cache[cache.family == "blue"].sort_values("k").scaled_integer.tolist()
)

display(cache.groupby("family").size().rename("exact terms").to_frame())
print("PASS: live derivation agrees with the full transparent cache")

In [ ]:
# Print compact sequence prefixes without converting large integers to floats.
sequence_prefixes = []
for family in ("red", "green", "yellow", "blue"):
    block = cache[cache.family == family].sort_values("k").head(6)
    for row in block.itertuples(index=False):
        sequence_prefixes.append({
            "family": family,
            "C": str(SCALES[family]),
            "k": int(row.k),
            "A_k": row.scaled_integer,
        })
display(pd.DataFrame(sequence_prefixes))

## 4. Recover mapped area from the integer period sequence

Because $\Phi_i=\Psi_i'$,

$$
\Psi_i(s)
 =\sum_{k\ge0}\frac{b_{i,k}}{k+1}s^{k+1}
 =\sum_{k\ge0}\frac{A_{i,k}}{k+1}\frac{s^{k+1}}{C_i^k}.
$$

Therefore

$$
\boxed{
\mathcal A_i(s)=\pi J_i(0)
\sum_{k\ge0}\frac{A_{i,k}}{k+1}\frac{s^{k+1}}{C_i^k}
}.
$$

This is the area/action relation already used in the paper and executable notebook; only the mapped-area weight $J_i$ and the additional Abel–Wick source planes are new.

In [ ]:
from validate_mapped_area_periods import (
    red_center_density,
    red_direct_quadrature,
    red_structured_mesh,
    aw_center_density,
    aw_direct_quadrature,
    aw_structured_mesh,
    mesh_area,
)

AW_INDEX = {"green": 0, "yellow": 1, "blue": 2}
CENTER_DENSITY = {
    "red": mp.mpf(str(red_center_density())),
    **{
        family: mp.mpf(str(aw_center_density(index)))
        for family, index in AW_INDEX.items()
    },
}


def area_from_integer_sequence(family: str, level: float, terms: int | None = None):
    block = cache[cache.family == family].sort_values("k")
    if terms is not None:
        block = block.head(terms)
    C = mp.mpf(str(SCALES[family]))
    s = mp.mpf(str(level))
    normalized = mp.mpf("0")
    for row in block.itertuples(index=False):
        k = int(row.k)
        A = mp.mpf(row.scaled_integer)
        normalized += A * s**(k + 1) / ((k + 1) * C**k)
    return mp.pi * CENTER_DENSITY[family] * normalized

## 5. Independent direct integral and mapped-triangle mesh

The sequence is checked against the original geometric definition

$$
\mathcal A_i(s)=\iint_{D_s}
\left\|R_{i,X}\times R_{i,Y}\right\|\,dX\,dY.
$$

The direct reference uses Gaussian quadrature of the exact Gram density.  The mesh check is independent: source vertices are mapped through the polynomial map and ordinary Euclidean triangle areas are summed in the three-dimensional range.  Coefficient data are not used by either calculation.

In [ ]:
TESTS = [
    ("red", 0.10),
    ("red", 0.40),
    ("green", 0.02),
    ("green", 0.05),
    ("yellow", 0.002),
    ("yellow", 0.005),
    ("blue", 0.002),
    ("blue", 0.005),
]
MESH_RESOLUTIONS = ((12, 48), (24, 96), (40, 160))

validation_rows = []
for family, level in TESTS:
    series_area = float(area_from_integer_sequence(family, level))
    if family == "red":
        direct_area = red_direct_quadrature(level, ntheta=240, nrho=72)
        mesh_builder = lambda nr, nt: red_structured_mesh(level, nr, nt)
    else:
        family_index = AW_INDEX[family]
        direct_area = aw_direct_quadrature(
            family_index, level, ntheta=240, nrho=72
        )
        mesh_builder = lambda nr, nt, index=family_index: aw_structured_mesh(
            index, level, nr, nt
        )

    row = {
        "family": family,
        "level": level,
        "sequence area": series_area,
        "direct Gram area": direct_area,
        "|sequence-direct|": abs(series_area - direct_area),
    }
    for nr, ntheta in MESH_RESOLUTIONS:
        _, vertices, triangles = mesh_builder(nr, ntheta)
        row[f"mesh {len(triangles)}"] = mesh_area(vertices, triangles)
    validation_rows.append(row)

validation = pd.DataFrame(validation_rows)
mesh_columns = [column for column in validation if column.startswith("mesh ")]
validation["finest relative mesh error"] = (
    (validation[mesh_columns[-1]] - validation["direct Gram area"]).abs()
    / validation["direct Gram area"].abs()
)
display(validation)

assert validation["|sequence-direct|"].max() < 2e-11
assert validation["finest relative mesh error"].max() < 8e-4
print("PASS: period sequence -> area -> direct Gram integral -> convergent mapped mesh")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for axis, family in zip(axes.ravel(), ("red", "green", "yellow", "blue")):
    family_rows = validation[validation.family == family]
    for _, row in family_rows.iterrows():
        triangle_counts = [int(column.split()[1]) for column in mesh_columns]
        errors = [
            abs(float(row[column]) - float(row["direct Gram area"]))
            for column in mesh_columns
        ]
        axis.loglog(triangle_counts, errors, marker="o", label=f"level={row['level']:g}")
    axis.set_title(f"{family}: mesh convergence")
    axis.set_xlabel("triangles")
    axis.set_ylabel("absolute area error")
    axis.grid(True, which="both", alpha=0.3)
    axis.legend()
fig.tight_layout()
plt.show()

In [ ]:
def draw_mapped_mesh(axis, vertices, triangles, color, title):
    faces = vertices[triangles]
    collection = Poly3DCollection(
        faces,
        facecolors=color,
        edgecolors=(0.1, 0.1, 0.1, 0.25),
        linewidths=0.22,
        alpha=0.82,
    )
    axis.add_collection3d(collection)
    lower = vertices.min(axis=0)
    upper = vertices.max(axis=0)
    center = 0.5 * (lower + upper)
    span = 0.58 * np.maximum(upper - lower, 1e-8)
    axis.set_xlim(center[0] - span[0], center[0] + span[0])
    axis.set_ylim(center[1] - span[1], center[1] + span[1])
    axis.set_zlim(center[2] - span[2], center[2] + span[2])
    axis.set_box_aspect(np.maximum(upper - lower, 1e-8))
    axis.view_init(elev=23, azim=-61)
    axis.set_title(title)

fig = plt.figure(figsize=(14, 11))
MESH_PANELS = [
    ("red", 0.40, "red"),
    ("green", 0.05, "green"),
    ("yellow", 0.005, "gold"),
    ("blue", 0.005, "royalblue"),
]
for panel, (family, level, color) in enumerate(MESH_PANELS, start=1):
    if family == "red":
        _, vertices, triangles = red_structured_mesh(level, 24, 96)
    else:
        _, vertices, triangles = aw_structured_mesh(
            AW_INDEX[family], level, 24, 96
        )
    axis = fig.add_subplot(2, 2, panel, projection="3d")
    draw_mapped_mesh(
        axis,
        vertices,
        triangles,
        color,
        f"{family}: level={level:g}, {len(triangles):,} triangles",
    )
fig.suptitle("Artifact 24 mapped-area meshes", fontsize=16)
fig.tight_layout()
plt.show()

## 6. Existing Hamilton–Abel depiction, extended by the Q/R coordinate net

The final interactive cell deliberately builds on the project’s earlier visual work:

- `artifact24.plots.build_interactive_figure` supplies the original domain/range closed-cycle figure;
- `notebooks/picture_candidates.ipynb` supplies the discriminant-surface viewpoint;
- `original_scripts/qr_coordinate_net_safe.py` overlays constant-$Q$ and constant-$R$ curves, their finite inverse images, and the pink missing cubic.

Inverse-map poles are masked with `np.divide(..., where=...)`, so excluded points are never evaluated.

In [ ]:
RUN_COORDINATE_NET = True
if RUN_COORDINATE_NET:
    runpy.run_path(
        str(ROOT / "original_scripts" / "qr_coordinate_net_safe.py"),
        run_name="__main__",
    )
else:
    print("Set RUN_COORDINATE_NET=True to build the rotatable Plotly figure.")

## 7. Result and reproducibility

This notebook has checked the following chain:

1. the red exact computation continues the earlier paper/notebook calculation;
2. green, yellow, and blue use the same bounded Gram-matrix method on the three Abel–Wick source planes;
3. a live exact prefix agrees with the long plain-text cache;
4. all cached rescaled coefficients are integers through the stated term counts;
5. yellow and blue agree term by term under reflection;
6. integrating the period sequence reproduces direct mapped area;
7. mapped triangle meshes converge to that same area;
8. numerical singularities are masked before division.

To regenerate a shorter cache visibly:

```bash
python original_scripts/build_mapped_area_period_cache.py \\
    --red-terms 12 --abel-wick-terms 12 --out /tmp/periods.csv
```

The 90/100-term cache was produced by the same scripts with larger term counts.  Integrality at finite depth remains evidence, not yet an all-orders theorem.